![Types of Attention](images/attention.png)

Self-attention in transformers is a technique designed to enhance input representations by enabling each position in a sequence to engage with and determine the relevance of every other position within the same sequence

![Self attention](images/self_attention_1.jpg)

![Self attention](images/self_attention_2.jpg)

To summarize `Self Attention` - we want to go from the simple `Input Embeddings` to a `Context Vector` for each token.

For a given token, we want to identify how much it is influenced by all the other tokens.

Hence, we will represent each token by a context vector, which is a weightd sum of all the other input elements

![Self attention example](images/self_attention_3.png)

![The process of computing attention vectors](images/self_attention_6.png)

## 1. Simplified Self Attention

The primary objective of this section is to demonstrate how the context vector 
Z(2) is calculated using the second input sequence, X(2), as a query

### 1.1 Simplified Self Attention for 1 sample input token

In [1]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/nn/modules/transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


In [2]:
# Step 1: As an example, compute the attention scores for one 
# of the inputs with all the other inputs

query = inputs[1]
attention_scores = torch.empty(inputs.shape[0])

for i, input in enumerate(inputs):
    attention_scores[i] = torch.dot(query, input)

print(attention_scores)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [3]:
# Step 2: Now, normalize the attention scores to get attention weights

# 1. Normalize by dividing by the sum of all weights
attention_weights_1 = attention_scores / attention_scores.sum()
print(attention_weights_1)
print(attention_weights_1.sum())

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
tensor(1.0000)


In [4]:
# However, in production, Softmax is preferred since it is better 
# at handling extreme values and has more desirable gradient properties 
# during training

attention_weights_2 = torch.softmax(attention_scores, dim=0)
print(attention_weights_2)
print(attention_weights_2.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [5]:
# Step 3: Compute the `context vector` for the selected input token
# by doing a weighted sum of all the other input vectors

context_vector_1 = torch.empty(query.shape)

for i, input in enumerate(inputs):
    context_vector_1 += input * attention_weights_2[i]

print(context_vector_1)

tensor([0.4419, 0.6515, 0.5683])


![computing the attention weight](images/self_attention_4.png)

### 1.2 Simplified Self-Attention for all Input Tokens

![Self Attention for all Input Tokens](images/self_attention_5.png)

![The process of computing attention vectors](images/self_attention_6.png)

In [6]:
# 1. Compute the attention scores

attention_scores = torch.empty(inputs.shape[0], inputs.shape[0])

for i, query in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attention_scores[i][j] = torch.dot(query, x_j)

print(attention_scores)
# the (i,j) th element is the dot product of token i with token j

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [7]:
# 2. Compute the attention weights

attention_weights = torch.empty(inputs.shape[0], inputs.shape[0])

for idx, attention_score in enumerate(attention_scores):
    attention_weights[idx] = torch.softmax(attention_score, dim=0)

print(attention_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [8]:
# 3. Compute the context vectors

# attn_weights = 6x6 matrix
# input = 6x3 matrix
# context_vector = 6x3 matrix (expected)
# So, we can just do attention_weights x input

# tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
#         [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
#         [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
#         [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
#         [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
#         [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

# inputs = torch.tensor(
#   [[0.43, 0.15, 0.89], # Your     (x^1)
#    [0.55, 0.87, 0.66], # journey  (x^2)
#    [0.57, 0.85, 0.64], # starts   (x^3)
#    [0.22, 0.58, 0.33], # with     (x^4)
#    [0.77, 0.25, 0.10], # one      (x^5)
#    [0.05, 0.80, 0.55]] # step     (x^6)
# )

context_vectors = attention_weights @ inputs
print(context_vectors)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


# 2. Self-attention with trainable weights

The previous section was for getting a "feel", this is the actual version used in GPT models and other LLMs.

* The overall idea is the same, we want to go from simple input embeddings (semantic + positional) to context vectors, which additionally encode how each token in the input relates to other tokens
* For each token, the context vector is a weighted sum over all the other input vectors
* The key **difference** is that we introduce some trainable weight matrices that allow the model to learn how to produce "good" context vectors.

To do this, we introduce three trainable matrices, `Wq`, `Wk` and `Wv` (query, key, value).

Then, each embedded input token is projected into the key, query and value vectors via matrix multiplication:
* Query vector (`qi`) = `xi * Wq`
* Key vector (`ki`) = `xi * Wk`
* Value vector (`vi`) = `xi * Wv`

*The embedding dimensions of the input and the query vector can be the same or different, depending on the model's design and specific implementation*

![Self attention overview](images/self_attention_7.jpg)

![Self attention overview](images/self_attention_8.jpg)

In [9]:
d_in = inputs.shape[1]
d_out = 2  # we're setting this to 2, that is, each input token will 
# be projected into a space of dimension 2

In [10]:
# Initialize the trainable KQV matrices

torch.manual_seed(123) # for consistent results

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

print(W_query)
print(W_key)
print(W_value)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])
Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])
Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


In [11]:
# Sample the values for the 2nd token just to get a "feel"

x_2 = inputs[1]
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key

print(query_2)
print(key_2)

tensor([0.4306, 1.4551])
tensor([0.4433, 1.1419])


In [12]:
# Compute the attention score of the second token with respect to the second token

attention_score_22 = torch.dot(query_2, key_2)
print(attention_score_22)

tensor(1.8524)


In [13]:
# Compute the attention score of the second token with respect to all other tokens

keys = inputs @ W_key

attention_scores_2 = query_2 @ keys.T
print(attention_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [14]:
# Compute the attention weights of the second token with respect to all other tokens

# The difference to earlier is that we now scale the attention scores by dividing them 
# by the square root of the embedding dimension, (i.e., d_k**0.5):

d_k = keys.shape[1]

attention_weights_2 = torch.softmax(attention_scores_2 / d_k**0.5, dim=-1)
print(attention_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [15]:
# Multiply the attention weights by the corresponding values to get 
# the context vector for the second token

values = inputs @ W_value

context_vector_2 = attention_weights_2 @ values
print(context_vector_2)

tensor([0.3061, 0.8210])


![Self attention overview](images/self_attention_9.png)

## 3. Self-attention with Trainable weights - Class

In [16]:
class SelfAttention_v1(torch.nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)

        context_vec = attn_weights @ values

        return context_vec


torch.manual_seed(123)
sa_v1 = SelfAttention_v1(3, 2)

print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


![Self Attention Summary](images/self_attention_10.png)

## Improving on the above - A Better Self Attention with Trainable Matrices class

Instead of `nn.Parameter`, let's use `nn.Linear` since it:
* has a preferred weight initialization scheme, which leads to more stable model training
* is equivalent to matrix multiplication if we disable the bias term

In [17]:
class SelfAttention_v2(torch.nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values =  self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)

        context_vec = attn_weights @ values

        return context_vec


torch.manual_seed(789)
sa_v2 = SelfAttention_v2(3, 2)

print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


# 3. Causal Self Attention (aka Masked Self-Attention)

`Self-attention => mask the future tokens => Causal self-attention`

In causal self attention, we make sure the input doesn't have access to the future tokens. 
We do this by masking the attention weights above the diagonal. 

As a result, the LLM is unable to utilize future tokens while calculating the context vectors from the attention weights.

*In simpler words, this ensures that each next word prediction should only depend on the preceding words*

In [18]:
# Let's look at the attention weights (without masking)

queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)

attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [19]:
# Let's create a mask using torch's lower triangle function

mask = torch.tril(torch.ones(attn_weights.shape[0], attn_weights.shape[0]))
print(mask)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [20]:
# Let's mask the attn_weight matrix using this

attn_weights_masked = attn_weights * mask
print(attn_weights_masked)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


## Issue: we need to re-normalize the attention weights after zeroing some of them

### Option 1: Normalize by dividing by the sum

![image](images/causal_attention_1.png)

In [21]:
rowwise_sum = attn_weights_masked.sum(dim=-1, keepdim=True)
attn_weights_masked_normalized = attn_weights_masked / rowwise_sum

print(attn_weights_masked_normalized)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


### Option 2: Mask with - infinity before applying Softmax

![image](images/causal_attention_2.png)

In [22]:
# Create an upper triangular mask of minus infinity

mask = torch.triu(torch.ones(attn_weights.shape[0], attn_weights.shape[0]), diagonal=1)
print(mask)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


In [23]:
# Apply the mask

attn_scores_masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(attn_scores_masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [24]:
# Verify that the attention weights in each row correctly sum to 1 again

attn_weights = torch.softmax(attn_scores_masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### Masking even more weights with Dropout

We want to avoid overfitting and the case where some neurons become *lazy*, so we additionally dropout some neurons (turn them off)

When we apply dropout, the non-dropped out values are scaled automatically (see example below)

![Dropout](images/causal_attention_3.png)

In [25]:
# An example of what drop-out looks like

torch.manual_seed(123)
sample_tensor = torch.ones(6,6)
dropout_layer = torch.nn.Dropout(0.5) # rate of 50%

resulting_tensor = dropout_layer(sample_tensor)
print(resulting_tensor)

# since we dropped out by 50%, the remaining values were scaled by a factor of 2
# the scaling is calculated by the formula 1 / (1 - dropout_rate)

tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])


In [26]:
# apply dropout to the attention weights above

torch.manual_seed(123)
attn_weights_with_dropout = dropout_layer(attn_weights)

print(attn_weights_with_dropout)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.8966, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4921, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4350, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<MulBackward0>)


In [53]:
class CausalAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout_fraction, kqv_bias=False):
        super().__init__()
        self.d_in = d_in  # say 3
        self.d_out = d_out  # say 2
  
        self.W_q = torch.nn.Linear(d_in, d_out, kqv_bias) # 2x3
        print(self.W_q.weight.shape)
  
        self.W_k = torch.nn.Linear(d_in, d_out, kqv_bias) # 2x3
     
        self.W_v = torch.nn.Linear(d_in, d_out, kqv_bias) # 2x3

        self.dropout = torch.nn.Dropout(dropout_fraction)
        
        upper_triangular_matrix = torch.triu(
            # 0 will mean allowed, 1 will mean mask out
            # diagnonal itself won't be masked, only the future to the right
            # [[0 1 1 1]
            #  [0 0 1 1]
            #  [0 0 0 1]
            #  [0 0 0 0]]
            torch.ones(context_length, context_length), diagonal=1
        )
        self.register_buffer("mask", upper_triangular_matrix)

    def forward(self, inputs):
        B, num_tokens, d_in = inputs.shape
        
        q = self.W_q(inputs) # B x T x d_out
        k = self.W_k(inputs)
        v = self.W_v(inputs)

        # q: B x T x d_out (batch x number of tokens x embedding dimension of k, q, v)
        # k.transpose(1,2): B x d_out x T
        # Hence, attn_scores will be B x T x T
        attn_scores = q @ k.transpose(1,2)
        # [:num_tokens, :num_tokens] crops the full context-length mask down to the actual sequence length
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = attn_weights @ v
        return context_vector

In [54]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

torch.Size([2, 6, 3])


In [55]:
torch.manual_seed(123)

context_length = batch.shape[1]
causal_attn = CausalAttention(
    d_in=3, d_out=2, context_length=context_length, dropout_fraction=0.0
)

context_vectors = causal_attn(batch)
print(context_vectors)

torch.Size([2, 3])
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
